# Cardiac Ultrasound Segmentation with U-Net | CAMUS

**Project goal:** Train a U-Net to identify three cardiac structures in echocardiography images: the **left-ventricular (LV) cavity**, **myocardium**, and **left atrium**. Each pixel is assigned one of four labels, including background.

**What this notebook covers:** dataset inspection → patient-level train/validation/test split → preprocessing → U-Net architecture → training → independent test evaluation → visual results.

**Dataset:** 450 CAMUS patients, 1,800 annotated 2D frames (two views, end-diastole and end-systole). **Held-out test set:** 68 patients / 272 frames. Input resolution: 256 × 256 grayscale.

**Important:** This notebook retains outputs from the completed Colab experiment. To reproduce training, download CAMUS from its official source, adjust `DATA_DIR`, and run the cells in order. A GPU is recommended. The trained checkpoint and cached arrays are stored separately in Google Drive, not embedded in the notebook.

**Labels:** `0` background · `1` LV cavity · `2` myocardium · `3` left atrium.

**Evaluation note:** Training uses *soft Dice* on probabilities; the final test results use *hard Dice* and IoU on predicted class masks. Background is excluded from the reported anatomical Dice/IoU averages. Pixel accuracy includes background.

## 1. Set up the environment

Install the medical-image reader and import the libraries. Google Drive holds the CAMUS files, preprocessed arrays, saved weights, and output figures. Update the paths if you keep the dataset elsewhere.

In [ ]:
# SimpleITK reads CAMUS .mhd medical-image files.
# Run this once in a fresh Google Colab session.
%pip install -q SimpleITK

In [ ]:
# Core libraries: numerical arrays, tables, image resizing, and plots.
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import SimpleITK as sitk
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from sklearn.model_selection import GroupShuffleSplit
import tensorflow as tf
from tensorflow.keras import layers, Model
from google.colab import drive

# Fix the random seed so splitting and model initialisation are reproducible.
# GPU operations may still show small run-to-run differences.
SEED = 42
IMAGE_SIZE = 256
NUM_CLASSES = 4
tf.keras.utils.set_random_seed(SEED)

# Connect to the same Drive folder used in the completed experiment.
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/ComputerVisionProject')
DATA_DIR = PROJECT_DIR / 'Original Camus Training Dataset' / 'training'
RESULTS_DIR = PROJECT_DIR / 'improved_unet_results'

# Fail early if the dataset path is wrong, rather than during training.
assert DATA_DIR.exists(), f'CAMUS dataset not found: {DATA_DIR}'
print('Total patient folders:', len(list(DATA_DIR.glob('patient*'))))
print('TensorFlow:', tf.__version__)
print('Available GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Inspect a CAMUS image and its expert mask

Before training, check that the ultrasound image aligns with its annotation and that the mask contains the four expected class labels. The overlay helps confirm that the annotation is in the correct location.

In [ ]:
# Choose one annotated frame from the first patient for a visual sanity check.
patient_dir = DATA_DIR / 'patient0001'
image_path = patient_dir / 'patient0001_2CH_ED.mhd'
mask_path = patient_dir / 'patient0001_2CH_ED_gt.mhd'

# SimpleITK loads medical-image volumes; squeeze removes singleton dimensions.
image = np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(str(image_path))))
mask = np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(str(mask_path))))
print('Image shape:', image.shape)
print('Mask shape:', mask.shape)
print('Mask labels:', np.unique(mask))

# Compare the raw ultrasound, the expert's pixel labels, and their overlay.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Echocardiography Image')
axes[1].imshow(mask, cmap='viridis', vmin=0, vmax=3)
axes[1].set_title('Ground Truth Segmentation')
axes[2].imshow(image, cmap='gray')
axes[2].imshow(np.ma.masked_where(mask == 0, mask), cmap='jet', alpha=0.4)
axes[2].set_title('Segmentation Overlay')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Pair images with masks and split patients

CAMUS supplies an ultrasound image and a corresponding expert-labelled mask. We match each pair using its filename. **Crucially, we split by patient ID**, not by image: frames from the same person must not appear in both training and testing, which would make the evaluation less independent.

In [ ]:
# Build one row per ultrasound image and matching ground-truth mask.
# The _gt suffix marks expert annotations in the CAMUS dataset.
records = []
for patient_dir in sorted(DATA_DIR.glob('patient*')):
    for mask_path in sorted(patient_dir.glob('*_gt.mhd')):
        image_path = mask_path.with_name(mask_path.name.replace('_gt.mhd', '.mhd'))
        if image_path.exists():
            records.append({
                'patient_id': patient_dir.name,
                'image_path': image_path,
                'mask_path': mask_path,
            })
df = pd.DataFrame(records)
print('Total annotated images:', len(df))
print('Total patients:', df['patient_id'].nunique())

# Resize images to a consistent shape and scale each image's intensities to [0, 1].
# Per-image normalisation helps account for ultrasound brightness differences.
def preprocess_image(path):
    image = np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(str(path)))).astype(np.float32)
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    image = (image - image.min()) / (image.max() - image.min() + 1e-8)
    return image[..., np.newaxis]  # Add the single grayscale channel.

# Masks are category IDs, not continuous brightness values.
# Nearest-neighbour resizing prevents invented labels between classes.
def preprocess_mask(path):
    mask = np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(str(path)))).astype(np.uint8)
    return cv2.resize(mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)

In [ ]:
# First reserve 30% of patients for validation + testing.
first_split = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(first_split.split(df, groups=df['patient_id']))
train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

# Divide the remaining patients into approximately equal validation/test groups.
second_split = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(second_split.split(temp_df, groups=temp_df['patient_id']))
val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

# These checks guard against patient leakage across the three splits.
train_patients = set(train_df['patient_id'])
val_patients = set(val_df['patient_id'])
test_patients = set(test_df['patient_id'])
assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

print('Dataset split completed successfully!\n')
for name, subset in [('Training', train_df), ('Validation', val_df), ('Testing', test_df)]:
    print(f'{name}: {len(subset)} images from {subset["patient_id"].nunique()} patients')

## 4. Preprocess the three datasets

Convert the matched files into NumPy arrays. Images have shape `(N, 256, 256, 1)` and masks `(N, 256, 256)`. Saving these arrays to Drive avoids repeating medical-image decoding every time Colab restarts.

In [ ]:
# Convert each split into arrays that TensorFlow can read efficiently.
def load_dataset(dataframe):
    images, masks = [], []
    for row in dataframe.itertuples(index=False):
        images.append(preprocess_image(row.image_path))
        masks.append(preprocess_mask(row.mask_path))

    X = np.stack(images).astype(np.float32)
    y = np.stack(masks).astype(np.uint8)
    return X, y

X_train, y_train = load_dataset(train_df)
X_val, y_val = load_dataset(val_df)
X_test, y_test = load_dataset(test_df)

# Confirm the expected input dimensions and valid anatomical labels.
for X, y in [(X_train, y_train), (X_val, y_val), (X_test, y_test)]:
    assert X.shape[1:] == (IMAGE_SIZE, IMAGE_SIZE, 1)
    assert y.shape[1:] == (IMAGE_SIZE, IMAGE_SIZE)
    assert set(np.unique(y)).issubset({0, 1, 2, 3})
print('Train / validation / test image shapes:', X_train.shape, X_val.shape, X_test.shape)

## 5. Build the four-class U-Net

U-Net has two halves: an **encoder** that learns increasingly complex patterns, and a **decoder** that restores the original image resolution. **Skip connections** pass fine spatial details from the encoder to the decoder, which helps locate anatomical boundaries. Batch normalisation stabilises training; spatial dropout regularises the deeper layers.

The final `softmax` layer produces four probabilities per pixel. Training combines **sparse cross-entropy** (correct class assignment) with **soft Dice loss** (overlap of anatomical regions). The Dice component ignores background.

In [ ]:
# Use the same architecture and loss as the completed training run.


# Two convolution layers with batch normalisation
def conv_block(inputs, filters, dropout=0.0):
    x = layers.Conv2D(
        filters, 3, padding="same",
        kernel_initializer="he_normal"
    )(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(
        filters, 3, padding="same",
        kernel_initializer="he_normal"
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    if dropout > 0:
        x = layers.SpatialDropout2D(dropout)(x)

    return x


# U-Net encoder-decoder with skip connections
def build_unet(input_shape=(256, 256, 1), num_classes=4):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128, 0.1)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256, 0.2)
    p4 = layers.MaxPooling2D()(c4)

    # Bottleneck
    bn = conv_block(p4, 512, 0.3)

    # Decoder
    u4 = layers.Conv2DTranspose(
        256, 2, strides=2, padding="same"
    )(bn)
    c5 = conv_block(
        layers.Concatenate()([u4, c4]), 256, 0.2
    )

    u3 = layers.Conv2DTranspose(
        128, 2, strides=2, padding="same"
    )(c5)
    c6 = conv_block(
        layers.Concatenate()([u3, c3]), 128, 0.1
    )

    u2 = layers.Conv2DTranspose(
        64, 2, strides=2, padding="same"
    )(c6)
    c7 = conv_block(
        layers.Concatenate()([u2, c2]), 64
    )

    u1 = layers.Conv2DTranspose(
        32, 2, strides=2, padding="same"
    )(c7)
    c8 = conv_block(
        layers.Concatenate()([u1, c1]), 32
    )

    outputs = layers.Conv2D(
        num_classes, 1, activation="softmax"
    )(c8)

    return Model(inputs, outputs, name="CAMUS_UNet")


# Soft Dice compares probability maps with expert masks during training.
# Exclude background so the large background area does not dominate the score.
def dice_score(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    y_true = tf.one_hot(y_true, depth=NUM_CLASSES)
    y_true = tf.cast(y_true, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred, axis=(0, 1, 2)
    )

    denominator = tf.reduce_sum(
        y_true + y_pred, axis=(0, 1, 2)
    )

    dice = (2 * intersection + 1e-6) / (
        denominator + 1e-6
    )

    return tf.reduce_mean(dice[1:])


# Cross-entropy penalises incorrect labels; Dice loss rewards spatial overlap.
def combined_loss(y_true, y_pred):
    ce = tf.keras.losses.sparse_categorical_crossentropy(
        y_true, y_pred
    )

    return tf.reduce_mean(ce) + (
        1.0 - dice_score(y_true, y_pred)
    )


# Compile with Adam; gradient clipping prevents unusually large updates.
model = build_unet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4,
        clipnorm=1.0
    ),
    loss=combined_loss,
    metrics=[dice_score]
)

print("Model:", model.name)
print("Parameters:", model.count_params())
print("Model successfully built and compiled!")

## 6. Run a quick model sanity check

Before committing to 60 epochs, confirm that two images produce `(2, 256, 256, 4)` probabilities and that the initial loss and Dice are finite. Low Dice before training is expected.

In [ ]:
# Test a small batch without updating the model weights.
predictions = model(X_train[:2], training=False)
loss_value = float(combined_loss(y_train[:2], predictions))
dice_value = float(dice_score(y_train[:2], predictions))
print('Prediction shape:', predictions.shape)
print('Initial loss:', round(loss_value, 4))
print('Initial Dice:', round(dice_value, 4))

# Catch numerical problems before a long GPU run.
assert predictions.shape == (2, IMAGE_SIZE, IMAGE_SIZE, NUM_CLASSES)
assert np.isfinite(loss_value) and np.isfinite(dice_value)
print('Model checks passed!')

## 7. Train and save the best model

Train for **up to 60 epochs** with batches of four images. A **checkpoint** saves the weights with the best validation soft Dice; **early stopping** avoids unnecessary epochs; and **learning-rate reduction** makes smaller updates when validation loss stops improving. A CSV log records each epoch so training curves can be recreated later.

**Recorded run:** 60 epochs completed, with the best validation checkpoint at epoch 59. Training is computationally expensive; rerunning this cell starts a new experiment and can overwrite the saved checkpoint.

In [ ]:
# Keep training artifacts in the dedicated results folder.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Select the best model using validation soft Dice, not the test set.
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(RESULTS_DIR / "best_unet.weights.h5"),
    monitor="val_dice_score",
    mode="max",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

# Stop after ten epochs without validation Dice improvement.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_dice_score",
    mode="max",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Halve the learning rate when validation loss stalls.
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Record per-epoch loss and Dice for later plots.
csv_logger = tf.keras.callbacks.CSVLogger(
    str(RESULTS_DIR / "training_history.csv")
)

# Fit on training images only; validation images monitor generalisation.
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=4,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr,
        csv_logger
    ],
    verbose=1
)

print("Training completed!")

## 8. Evaluate the held-out test patients

Load the **best validation checkpoint** and predict the 272 images belonging to patients never used in training or checkpoint selection. `argmax` chooses the most likely class at each pixel.

For each anatomical class, calculate **Dice** (overlap relative to mask sizes) and **IoU** (intersection divided by union), then average across images. Empty/empty masks are excluded from that class's average rather than counted as perfect predictions. The macro score gives equal weight to the three anatomical classes.

In [ ]:
# Restore the best validation checkpoint before testing.
model.load_weights(str(RESULTS_DIR / "best_unet.weights.h5"))

CLASS_NAMES = {
    1: "LV cavity",
    2: "Myocardium",
    3: "Left atrium"
}

# Predict in batches of four to reduce GPU memory use.
predicted_masks = []

for start in range(0, len(X_test), 4):
    probabilities = model.predict(
        X_test[start:start + 4],
        verbose=0
    )
    predicted_masks.append(
        np.argmax(probabilities, axis=-1).astype(np.uint8)
    )

y_pred = np.concatenate(predicted_masks, axis=0)

# Measure region overlap separately for each image and each structure.
rows = []

for i in range(len(y_test)):
    for class_id, class_name in CLASS_NAMES.items():
        true_mask = y_test[i] == class_id
        pred_mask = y_pred[i] == class_id

        intersection = np.logical_and(
            true_mask, pred_mask
        ).sum()

        true_count = true_mask.sum()
        pred_count = pred_mask.sum()
        union = np.logical_or(true_mask, pred_mask).sum()

        # An absent structure in both masks has no defined overlap, so exclude it from that class average.
        dice = (
            2 * intersection / (true_count + pred_count)
            if true_count + pred_count > 0 else np.nan
        )

        iou = (
            intersection / union
            if union > 0 else np.nan
        )

        rows.append({
            "image_index": i,
            "class": class_name,
            "dice": dice,
            "iou": iou
        })

results = pd.DataFrame(rows)

summary = results.groupby("class")[["dice", "iou"]].agg(
    ["mean", "std", "count"]
)

print("\nFINAL TEST RESULTS")
print(summary.round(4))

print("\nMean anatomical Dice:",
      round(results.groupby("class")["dice"].mean().mean(), 4))

print("Mean anatomical IoU:",
      round(results.groupby("class")["iou"].mean().mean(), 4))

# Save per-image results, class summaries, and discrete prediction masks.
results.to_csv(RESULTS_DIR / "test_per_image_metrics.csv", index=False)
summary.to_csv(RESULTS_DIR / "test_summary_metrics.csv")

np.save(RESULTS_DIR / "test_predictions.npy", y_pred)

# Reuse the already-computed test predictions to avoid duplicate GPU inference.
# Each pixel is correct when its predicted class matches the expert label.
correct_pixels = np.count_nonzero(y_pred == y_test)
total_pixels = y_test.size
pixel_accuracy = 100.0 * correct_pixels / total_pixels
print(f'Test Pixel Accuracy: {pixel_accuracy:.2f}%')
print("\nTest results saved successfully!")

## 9. Visualise predictions against expert annotations

Numbers alone cannot show where boundaries fail. These four representative held-out frames display the ultrasound, expert mask, model prediction, and a transparent overlay. The same class colours are used across the rows.

In [ ]:
# Choose four fixed test indices for a repeatable visual comparison.
indices = [0, 50, 100, 150]  # Fixed examples from the recorded 272-frame test split.

# Keep class colours consistent between expert masks and predictions.
cmap = ListedColormap([
    "#111827",  # Background
    "#2563eb",  # LV cavity
    "#10b981",  # Myocardium
    "#f59e0b"   # Left atrium
])
norm = BoundaryNorm(np.arange(-0.5, 4.5, 1), cmap.N)

fig, axes = plt.subplots(4, 4, figsize=(15, 15))

for row, idx in enumerate(indices):
    # The same image is shown with its reference and predicted labels.
    image = X_test[idx, :, :, 0]
    actual = y_test[idx]
    predicted = y_pred[idx]

    axes[row, 0].imshow(image, cmap="gray")
    axes[row, 1].imshow(actual, cmap=cmap, norm=norm)
    axes[row, 2].imshow(predicted, cmap=cmap, norm=norm)

    axes[row, 3].imshow(image, cmap="gray")
    axes[row, 3].imshow(
        np.ma.masked_where(predicted == 0, predicted),
        cmap=cmap,
        norm=norm,
        alpha=0.5
    )

    for ax in axes[row]:
        ax.axis("off")

for ax, title in zip(
    axes[0],
    ["Ultrasound", "Ground Truth", "U-Net Prediction", "Prediction Overlay"]
):
    ax.set_title(title, fontsize=13, fontweight="bold")

plt.tight_layout()

# Save a publication-ready figure for the project README.
figure_path = RESULTS_DIR / "segmentation_predictions.png"
plt.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", figure_path)

## 10. Plot training and validation curves

These curves show how soft Dice and loss changed over the training run. Training-versus-validation differences help assess generalisation; the checkpoint is chosen from **validation Dice**, not from the test set.

In [ ]:
# Read the per-epoch CSV saved during training.
history_df = pd.read_csv(
    RESULTS_DIR / "training_history.csv"
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Compare training and validation soft Dice across epochs.
axes[0].plot(
    history_df["dice_score"],
    label="Training Dice",
    linewidth=2
)
axes[0].plot(
    history_df["val_dice_score"],
    label="Validation Dice",
    linewidth=2
)
axes[0].set_title("Segmentation Dice Score")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Dice Score")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Compare training and validation loss across epochs.
axes[1].plot(
    history_df["loss"],
    label="Training Loss",
    linewidth=2
)
axes[1].plot(
    history_df["val_loss"],
    label="Validation Loss",
    linewidth=2
)
axes[1].set_title("Training and Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()

# Save the chart for the GitHub README.
plt.savefig(
    RESULTS_DIR / "training_curves.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Training curves saved successfully!")

## Results and interpretation

**Recorded results from the completed four-class CAMUS experiment (68 held-out patients; 272 test frames)**

| Test metric | Result |
|---|---:|
| LV cavity Dice | **92.62%** |
| Myocardium Dice | **85.65%** |
| Left atrium Dice | **88.14%** |
| **Mean anatomical Dice** | **88.80%** |
| **Mean anatomical IoU** | **80.77%** |

The LV cavity has the highest mean Dice. The left atrium has the greatest variation in Dice across test frames (standard deviation 0.1262). The reported Dice and IoU averages exclude background. Pixel accuracy, calculated by the evaluation cell, includes background and should be interpreted separately.

**Dataset reference:** Leclerc et al. (2019), *Deep Learning for Segmentation Using an Open Large-Scale Dataset in 2D Echocardiography*.
